<a href="https://colab.research.google.com/github/akbarafriansyah/data-science-2026/blob/main/Pertemuan12_Afriansyah_Akbar_250401020013.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Nama Mahasiswa	: Afriansyah Akbar, A.Md

NIM		: 250401020013

Kelas    : IF401

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)
import pandas as pd, numpy as np
import matplotlib.pyplot as plt

print (" ")
print ("1: Generate & Eksplorasi Dataset Transaksi")
print (" ")
np.random.seed(42)
produk = ['Roti', 'Selai', 'Susu', 'Sereal', 'Telur',
'Keju', 'Kopi', 'Gula', 'Teh', 'Mentega']
# Buat 50 transaksi, tiap transaksi berisi 2-5 produk
transaksi = []
for _ in range(50):
  n_item = np.random.randint(2, 6)
  transaksi.append(list(np.random.choice(produk, n_item, replace=False)))
# Suntikkan pola: Roti sering bersama Selai
for i in range(0, 20):
  if 'Roti' in transaksi[i] and 'Selai' not in transaksi[i]:
    transaksi[i].append('Selai')

print('Contoh transaksi:', transaksi[:3])
print('Jumlah transaksi:', len(transaksi))

print (" ")
print ("2: One-Hot Encoding Transaksi")
print (" ")
from mlxtend.preprocessing import TransactionEncoder
te = TransactionEncoder()
te_ary = te.fit(transaksi).transform(transaksi)
df = pd.DataFrame(te_ary, columns=te.columns_)
from mlxtend.frequent_patterns import apriori
for ms in [0.05, 0.1, 0.2]:
  freq = apriori(df, min_support=ms, use_colnames=True)
print(f'min_support={ms}: {len(freq)} itemset ditemukan') # Gunakan min_support yang menghasilkan jumlah itemset wajar (tidak 0, tidak ratusan)
freq_items = apriori(df, min_support=0.1, use_colnames=True)
freq_items = freq_items.sort_values('support', ascending=False)
print(freq_items.head(10))
print(df.head())

print (" ")
print ("3: Cari Frequent Itemset dengan Apriori")
print (" ")
from mlxtend.frequent_patterns import apriori
for ms in [0.05, 0.1, 0.2]:
  freq = apriori(df, min_support=ms, use_colnames=True)
print(f'min_support={ms}: {len(freq)} itemset ditemukan') # Gunakan min_support yang menghasilkan jumlah itemset wajar (tidak 0, tidak ratusan)
freq_items = apriori(df, min_support=0.1, use_colnames=True)
freq_items = freq_items.sort_values('support', ascending=False)
print(freq_items.head(10))

print (" ")
print ("4: Bentuk & Saring Aturan Asosiasi")
print (" ")
from mlxtend.frequent_patterns import association_rules
rules = association_rules(freq_items, metric='confidence',
min_threshold=0.5)
rules = rules[rules['lift'] > 1].sort_values('lift', ascending=False)
print(rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(10))

print ("Interpretasikan dalam sel Markdown:")
print ("Aturan mana yang paling kuat (Lift tertinggi)?    : Aturan yang melibatkan Roti dan Selai ")
print ("Apakah masuk akal secara bisnis (mis. Roti -> Selai)?   : Ya, aturan tersebut masuk akal secara bisnis karena pelanggan yang membeli roti sering juga membeli selai sebagai pelengkap, sehingga kedua produk memiliki keterkaitan pembelian yang kuat")


print (" ")
print ("5: Rekomender Sederhana dengan Content-Based Filtering")
print (" ")
from sklearn.metrics.pairwise import cosine_similarity
katalog = pd.DataFrame({
'produk': produk,
'kategori': ['Bakery','Bakery','Dairy','Bakery','Dairy',
'Dairy','Minuman','Bumbu','Minuman','Dairy']
})
fitur = pd.get_dummies(katalog['kategori'])
sim_matrix = cosine_similarity(fitur)
def rekomendasi_serupa(nama_produk, top_n=3):
  idx = katalog.index[katalog['produk'] == nama_produk][0]
  skor = list(enumerate(sim_matrix[idx]))
  skor = sorted(skor, key=lambda x: x[1], reverse=True)
  skor = [s for s in skor if s[0] != idx][:top_n]
  return katalog.iloc[[i for i, _ in skor]]['produk'].tolist()

print('Mirip dengan Roti:', rekomendasi_serupa('Roti'))

print (" ")
print ("6: Bandingkan Kedua Pendekatan")
print (" ")
produk_target = 'Roti'
rules_terkait = rules[rules['antecedents'].apply(lambda x: produk_target in x)]
print('Rekomendasi dari Association Rules:')
print(rules_terkait[['consequents', 'lift']].head())
print('Rekomendasi dari Content-Based:', rekomendasi_serupa(produk_target))


 
1: Generate & Eksplorasi Dataset Transaksi
 
Contoh transaksi: [[np.str_('Keju'), np.str_('Roti'), np.str_('Mentega'), np.str_('Kopi'), 'Selai'], [np.str_('Roti'), np.str_('Kopi'), np.str_('Teh'), np.str_('Selai'), np.str_('Mentega')], [np.str_('Kopi'), np.str_('Susu'), np.str_('Teh')]]
Jumlah transaksi: 50
 
2: One-Hot Encoding Transaksi
 
min_support=0.2: 13 itemset ditemukan
    support      itemsets
5      0.52       (Selai)
8      0.46         (Teh)
3      0.42     (Mentega)
9      0.36       (Telur)
1      0.34        (Keju)
0      0.32        (Gula)
2      0.32        (Kopi)
4      0.32        (Roti)
7      0.32        (Susu)
36     0.24  (Selai, Teh)
    Gula   Keju   Kopi  Mentega   Roti  Selai  Sereal   Susu    Teh  Telur
0  False   True   True     True   True   True   False  False  False  False
1  False  False   True     True   True   True   False  False   True  False
2  False  False   True    False  False  False   False   True   True  False
3  False   True  False    False

-  Apakah kedua pendekatan memberi rekomendasi yang konsisten?
-  Kapan sebaiknya menggunakan salah satu, atau menggabungkan keduanya (hybrid)?

    ==============================================

1. Tidak selalu. Pada contoh ini keduanya dapat merekomendasikan produk yang berkaitan dengan Roti, tetapi dasar rekomendasinya berbeda. Association Rules berdasarkan pola pembelian pelanggan (misalnya Roti → Selai), sedangkan Content-Based Filtering berdasarkan kemiripan kategori produk (misalnya Roti → Sereal atau Selai karena sama-sama kategori Bakery).

2. **Association Rules** digunakan ketika tersedia banyak data transaksi dan ingin mengetahui produk yang sering dibeli bersama untuk kebutuhan cross-selling.
**Content-Based Filtering **digunakan ketika data transaksi masih terbatas atau terdapat produk baru yang belum memiliki riwayat pembelian.
**Hybrid** digunakan untuk memperoleh rekomendasi yang lebih akurat dengan menggabungkan pola pembelian pelanggan dan kemiripan karakteristik produk.